# 01. Symbolic Allen core

This is the bottom layer of `alpha_rule`: Allen interval relations, the matrix
form of a rule, and matching a rule against a history of events. It depends only
on numpy. Everything else in the project sits on top of it.

Two subpackages:

* `alpha_rule.helpers` holds the `Event` triple and the Allen matrix math (the
  `AllenRelation` enum, the `(n+2, n)` matrix validator, and the converters
  between a flat rule string and a matrix).
* `alpha_rule.rules` wraps a matrix as `AllenMatrix` and matches a rule against a
  history of events.

In [ ]:
import alpha_rule
print("alpha_rule", alpha_rule.__version__)

## Elements

`alpha_rule.helpers.matrix_operations`
* `AllenRelation`: the 13 Allen relations as an enum. `allen_relations` is the
  list of their symbols.
* `validate_standard_matrix(m)`: checks the `(n+2, n)` shape, the indicator row,
  the `=` diagonal, and that every cell is a known relation or `#`.
* `hierarchy_string_to_matrix(s)` and `matrix_to_hierarchy_string(m)`: convert
  between a flat rule string like `"A B <"` and the matrix.

`alpha_rule.helpers.generic`
* `Event(type, start, end)`: one temporal event.

`alpha_rule.rules.allen_matrix`
* `AllenMatrix`: a validated matrix. Build one from a string with
  `AllenMatrix.from_hierarchy_string("A B <")` and read it back with
  `.get_hierarchy_string()`.

`alpha_rule.rules.rule_matching`
* `determine_allen_relation(e1, e2)`: the relation symbol between two events.
* `generate_allen_matrix_from_history(history)`: build the matrix for a list of
  events (the history is stored newest first).
* `apply_binary_vector(matrix, vec)`: keep only the events marked `1` in `vec`,
  giving a smaller pattern matrix.
* `match_rule_to_history(rule_matrix, history)`: does the rule match a list of
  events (most recent last)?
* `match_rule_to_matrix(rule_matrix, history_matrix)`: the same test against a
  prebuilt history matrix (a raw numpy array) instead of an event list.

### The 13 relations

In [ ]:
from alpha_rule.helpers.matrix_operations import AllenRelation, allen_relations

print("count:", len(allen_relations))
print("symbols:", allen_relations)
print("BEFORE ->", AllenRelation.BEFORE.value)

### A rule as a matrix

A rule string and its matrix are two views of the same thing.

In [ ]:
from alpha_rule.rules.allen_matrix import AllenMatrix

rule = AllenMatrix.from_hierarchy_string("A B <")   # a 2 event rule: A before B
print(rule)
print("back to string:", rule.get_hierarchy_string())

### A matrix from an event history

Instead of a string, you can build the matrix straight from a list of events.
The history is stored newest first, so the top type row reads `W Z Y X` for a
history given oldest first.

In [ ]:
from alpha_rule.helpers.generic import Event
from alpha_rule.rules.rule_matching import generate_allen_matrix_from_history

history = [Event("X", 1, 5), Event("Y", 4, 8), Event("Z", 7, 10), Event("W", 10, 13)]
print(generate_allen_matrix_from_history(history))

### The relation between two events

In [ ]:
from alpha_rule.rules.rule_matching import determine_allen_relation

print("A ends as B starts:", determine_allen_relation(Event("A", 0, 5), Event("B", 5, 10)))   # m, meets
print("A well before B:   ", determine_allen_relation(Event("A", 0, 1), Event("B", 5, 6)))    # <, before
print("same span:         ", determine_allen_relation(Event("A", 0, 5), Event("B", 0, 5)))    # =, equals

### Matching a rule against a history

The matrix stores the history newest first, so `"B A <"` reads as: the last
event is `B`, and an earlier `A` came before it.

In [ ]:
from alpha_rule.rules.rule_matching import match_rule_to_history

rule = AllenMatrix.from_hierarchy_string("B A <")
print("A then B  :", match_rule_to_history(rule, [Event("A", 0, 1), Event("B", 5, 6)]))   # True
print("A then A  :", match_rule_to_history(rule, [Event("A", 0, 1), Event("A", 5, 6)]))   # False, last event is not B

## Basic checks

A handful of asserts that mirror the unit tests. Run the cell. If it prints
`ok`, the component behaves as expected.

In [ ]:
# Rule strings round trip through the matrix.
for s in ["A", "A B <", "A B < C < <", "A A ="]:
    assert AllenMatrix.from_hierarchy_string(s).get_hierarchy_string() == s, s

# A trailing <END> marker is stripped before parsing.
assert AllenMatrix.from_hierarchy_string("A B < <END>").get_hierarchy_string() == "A B <"

# Every relation symbol is one or two characters.
assert all(1 <= len(v) <= 2 for v in allen_relations)

# determine_allen_relation on three canonical cases.
assert determine_allen_relation(Event("A", 0, 1), Event("B", 5, 6)) == "<"
assert determine_allen_relation(Event("A", 0, 5), Event("B", 5, 10)) == "m"
assert determine_allen_relation(Event("A", 0, 5), Event("B", 0, 5)) == "="

# Matching: a positive case, a wrong last event, and a history that is too short.
assert match_rule_to_history(AllenMatrix.from_hierarchy_string("A"),
                             [Event("B", 0, 1), Event("A", 2, 3)]) is True
assert match_rule_to_history(AllenMatrix.from_hierarchy_string("A"),
                             [Event("A", 0, 1), Event("B", 2, 3)]) is False
assert match_rule_to_history(AllenMatrix.from_hierarchy_string("A B <"),
                             [Event("B", 0, 1)]) is False

print("ok")

### Self consistency

Any pattern you pull out of a history should still match that history. Build a
random history, extract every sub pattern of up to four events with
`apply_binary_vector`, and confirm they all match. This is a strong property:
the matcher must find a pattern that came from the history itself.

In [ ]:
import random
import itertools
from alpha_rule.rules.rule_matching import apply_binary_vector

event_types = ["A", "B", "C", "D"]
random.seed(42)

def generate_random_history(num_events):
    history = []
    current_time = 0
    for _ in range(num_events):
        duration = random.randint(1, 15)
        start = current_time
        end = start + duration
        history.append(Event(random.choice(event_types), start, end))
        current_time = end + random.randint(1 - duration, 2)
    return history

def patterns_up_to(n, max_events):
    # binary vectors of length n with position 0 always set and at most
    # max_events ones in total.
    for extra in range(0, max_events):
        for combo in itertools.combinations(range(1, n), extra):
            vec = [0] * n
            vec[0] = 1
            for p in combo:
                vec[p] = 1
            yield vec

hist = generate_random_history(12)
full = generate_allen_matrix_from_history(hist)
vectors = list(patterns_up_to(len(hist), max_events=4))
matched = sum(match_rule_to_history(apply_binary_vector(full, v), hist) for v in vectors)

print(f"{matched} of {len(vectors)} extracted patterns match their own history")
assert matched == len(vectors)
print("ok")

## Timing

The matcher is fast across the whole grid, including the 100 event, length 10
cell. `match_rule_to_matrix` is a touch faster than `match_rule_to_history`
because it matches a prebuilt matrix instead of rebuilding it from the event list
on every call. Realistic histories are smaller still: the wrapper keeps a window
of about 15 to 20 events, where every cell is sub millisecond.

In [ ]:
import time
from alpha_rule.rules.rule_matching import match_rule_to_matrix

history_sizes = [10, 20, 50, 100]
pattern_sizes = [3, 5, 10]
SEED = 0

def select_random_pattern_from_history(history, size):
    size = size - 1
    indices = sorted(random.sample(range(len(history)), size))
    vec = [1] + [1 if i in indices else 0 for i in range(len(history) - 1)]
    return apply_binary_vector(generate_allen_matrix_from_history(history), vec)

def run_timing_grid(seed, use_matrix):
    random.seed(seed)
    for h_size in history_sizes:
        history = generate_random_history(h_size)
        history_matrix = generate_allen_matrix_from_history(history).matrix if use_matrix else None
        for p_size in pattern_sizes:
            if p_size > h_size:
                continue
            pattern = select_random_pattern_from_history(history, p_size)
            start = time.perf_counter()
            if use_matrix:
                matched = match_rule_to_matrix(pattern, history_matrix)
            else:
                matched = match_rule_to_history(pattern, history)
            end = time.perf_counter()
            print({
                "history_size": h_size,
                "pattern_size": p_size,
                "matched": matched,
                "time_seconds": end - start,
            })

### match_rule_to_history

Matches a list of events, rebuilding the history matrix on every call.

In [ ]:
run_timing_grid(SEED, use_matrix=False)

### match_rule_to_matrix

Matches against a prebuilt history matrix (the raw array the wrapper keeps up to
date), so it skips the rebuild that `match_rule_to_history` pays. Same grid, same
match result.

In [ ]:
run_timing_grid(SEED, use_matrix=True)

## Incremental reuse

This is why `match_rule_to_matrix` takes a prebuilt matrix. In a stream, events
arrive one at a time and you match after each one. `match_rule_to_history` rebuilds
the whole `(n+2, n)` matrix from the event list on every call, recomputing every
pairwise Allen relation. If instead you keep the matrix and extend it as each event
arrives, computing only the new event's relations and reusing the rest, you hand it
straight to `match_rule_to_matrix` and skip the full recompute. This is what the
wrapper does (component 07).

The cell below writes a small incremental updater, checks it matches a full rebuild
exactly, then times rebuild-every-step against grow-incrementally. The maintained
version is a few times faster.

In [ ]:
import numpy as np
from alpha_rule.helpers.matrix_operations import check_rows_columns_combined

def append_event_to_matrix(matrix, prior_events, new_event):
    """Extend a newest-first history matrix by one event, computing only the new
    event's relations and copying the rest, instead of recomputing every relation
    from scratch. prior_events is the events seen so far, oldest first, not yet
    including new_event."""
    if matrix is None or matrix.shape[1] == 0:
        m = np.full((3, 1), "#", dtype=object)
        m[1, 0] = new_event.type
        m[2, 0] = "="
        m[0] = check_rows_columns_combined(m[1:]).astype(int)
        return m
    n_old = matrix.shape[1]
    n_new = n_old + 1
    new = np.full((n_new + 2, n_new), "#", dtype=object)
    new[1, 0] = new_event.type                       # newest event type at column 0
    new[1, 1:] = matrix[1, :]                         # older types shift right
    new[3:n_new + 2, 1:n_new] = matrix[2:n_old + 2, 0:n_old]   # old relations shift diagonally
    for j in range(1, n_new):                         # relations of new event vs each older one
        new[2, j] = determine_allen_relation(prior_events[n_old - j], new_event)
    for i in range(n_new):                            # diagonal
        new[i + 2, i] = "="
    new[0] = check_rows_columns_combined(new[1:]).astype(int)
    return new

# Correctness: the incrementally grown matrix equals a full rebuild at every step.
random.seed(0)
stream = generate_random_history(60)
matrix, events = None, []
for ev in stream:
    matrix = append_event_to_matrix(matrix, events, ev)
    events.append(ev)
    assert np.array_equal(matrix, generate_allen_matrix_from_history(events).matrix)
print("incremental matrix matches a full rebuild at every step: ok")

In [ ]:
# Stream events in one at a time and match a fixed rule after each one.
rule = AllenMatrix.from_hierarchy_string("B A <")

for N in (50, 100, 200):
    random.seed(1)
    stream = generate_random_history(N)

    # Rebuild the whole matrix on every new event (what match_rule_to_history does).
    t0 = time.perf_counter()
    for t in range(1, N + 1):
        m = generate_allen_matrix_from_history(stream[:t]).matrix
        match_rule_to_matrix(rule, m)
    rebuild = time.perf_counter() - t0

    # Grow the matrix incrementally and reuse it.
    t0 = time.perf_counter()
    matrix, events = None, []
    for ev in stream:
        matrix = append_event_to_matrix(matrix, events, ev)
        events.append(ev)
        match_rule_to_matrix(rule, matrix)
    incremental = time.perf_counter() - t0

    print(f"stream={N:3d} events   rebuild each step={rebuild:6.3f}s   "
          f"grow incrementally={incremental:6.3f}s   speedup={rebuild / incremental:4.1f}x")

## Full unit tests

The cells above are a quick tour. The full suite for this component lives in the
project and covers all 13 relations, the round trip and shape contracts, and the
type aware matcher. Run it from the project folder:

```
python -m alpha_rule.tests.run_tests test_matrix_operations
python -m alpha_rule.tests.run_tests test_allen_matrix
python -m alpha_rule.tests.run_tests test_rule_matching
python -m alpha_rule.tests.run_tests test_type_aware_enum
```